# Install Dependencies

In [ ]:
!pip install -q bitsandbytes
!pip install -q datasets
!pip install -q peft
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 4.2 MB/s eta 0:00:00


# login

In [ ]:
import huggingface_hub
huggingface_hub.login(Hugging_Face_TOKEN)

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import random
import pandas as pd
import re
import string
import sys
import argparse
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
from sklearn.utils import shuffle
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import emoji
import pyarabic.araby as araby

# Define Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['Positive', 'Negative', 'Neutral']
    mapping = {'Positive': 0, 'Negative': 1, 'Neutral':2, 'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred, digits = 4)
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2, 3])
    print('\nConfusion Matrix:')
    print(conf_matrix)

# Define Predict Function

In [ ]:
def predict(X_test, model, tokenizer):
    y_pred = []
    max_length = tokenizer.model_max_length
    for i in tqdm(range(len(X_test))):
        prompt = X_test.iloc[i]["Text"]
        prompt = prompt[:max_length]
        result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
        answer = result[0]['generated_text'].split("=")[-1].lower()
        if "positive" in answer:
            y_pred.append("Positive")
        elif "negative" in answer:
            y_pred.append("Negative")
        elif "neutral" in answer:
            y_pred.append("Neutral")
        else:
            y_pred.append("none")
    return y_pred

# Load the Data

In [ ]:
data1 = pd.read_csv('train_all.csv')
ind = pd.read_excel('Used ASAD For Training.xlsx')

data = data1.loc[ind['Unnamed: 0'].values]
data = data[["Text", "sentiment"]]

data.columns = ["Text", "sentiment"]
print(data["sentiment"].value_counts())

sentiment
neutral     16289
negative     3890
positive     3778
Name: count, dtype: int64


In [ ]:
data.shape

(23957, 2)

# Preprocessing

In [ ]:
data['Text'] = data['Text'].str.replace(r'[^\w\s]+', '')
data['Text'] = data['Text'].str.replace("\s+", " ", regex=True)

data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو إن الأسى يأكلك بأكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا أمتلك تعريفا واضحا للإهاب بس طالما جامعة ال...,neutral


In [ ]:
arabic_punctuations = '''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~¦+|!”…“–ـ'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations

def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("گ", "ك", text)
    return text

In [ ]:
data['Text'] = data['Text'].apply(normalize_arabic)
data['Text'] = data['Text'].apply(remove_repeating_char)

data.head()

,Text,sentiment
48348,ياحبّي المغرور ياللي دِفاك شعور رد القمر للنور...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
def data_cleaning(text):
    """Clean and preprocess text data.
    Args:
        text (pd.Series): A pandas Series containing text data to be cleaned.
    Returns:
        pd.Series: A pandas Series with the cleaned text data.

    Cleaning Steps:
    - Removes emojis and special characters like '\x89Û_', '&amp', etc.
    - Replaces consecutive dots with an empty string.
    - Removes '#' symbol from text.
    - Removes user names starting with '@'.
    - Removes URLs starting with 'http' or 'https'.
    - Remove diacritics.
    - Remove English.
    - fix elongated words
    - Normalize Characters
    - Removes extra whitespaces between words.

    """
    clean = text
    # Replace consecutive dots with an empty string
    pattern = re.compile('\\.+?(?=\B|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Replace '\x89Û_' with a whitespace
    pattern = re.compile('\x89Û_')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Replace newline characters with a whitespace
    pattern = re.compile('\\n')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=' '))
    # Remove '#' symbol from text
    clean = clean.apply(lambda r: r.replace('#', ''))
    # Remove '_' symbol from text
    pattern = re.compile('_')
    clean = clean.apply(lambda r: re.sub(pattern, ' ', r))
    # Replace user names with '@'
    pattern = re.compile('@[a-zA-Z0-9\_]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='@'))
    # Remove URLs
    pattern = re.compile('https?\S+(?=\s|$)')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl='www'))
    # Remove emojis
    clean = clean.apply(lambda r: emoji.replace_emoji(r, replace=""))
    # Remove diacritics
    clean = clean.apply(lambda r: araby.strip_diacritics(r))
    # Remove English
    # pattern = re.compile(r'[a-zA-Z]+')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=''))
    # Fix elongated words
    pattern = re.compile(r'(.)\1{2,}')
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl=r'\1'))
    # Normalize letters
    pattern = re.compile("[إأآا]")
    clean = clean.apply(lambda r: re.sub(pattern, string=r, repl="ا"))
    # Remove extra whitespaces
    clean = clean.apply(lambda r: ' '.join(r.split()))  # Remove extra whitespaces between words

    return clean

In [ ]:
data['Text'] = data_cleaning(data['Text'])

In [ ]:
def remove_ids(text):
  return text.split("—")[0].strip()

data['Text'] = data['Text'].apply(remove_ids)
data.head()

,Text,sentiment
48348,ياحبي المغرور ياللي دفاك شعور رد القمر للنور و...,positive
47523,من شدة حبك له تتمنى لو ان الاسى ياكلك باكملك و...,positive
2695,يالليل يا جامع على الود قلبين,neutral
54022,حفلة اكثر من رائعة ستقام في جوهرة جدة كبيرة وج...,positive
7247,لا امتلك تعريفا واضحا للاهاب بس طالما جامعة ال...,neutral


In [ ]:
data.isnull().sum()

,0
Text,0
sentiment,0


In [ ]:
data.dropna(inplace = True)
data = data.drop_duplicates(subset='Text')
data.shape

(23957, 2)

In [ ]:
data = data.reset_index(drop = True)

# Split Data

In [ ]:
indecies = pd.read_csv('Train-Val-Test-Indecies-Climate-ASAD.csv')

# Clean and convert to integer index arrays
test = data.loc[indecies['test'].dropna().astype(int).values]
train = data.loc[indecies['train'].dropna().astype(int).values]
val = data.loc[indecies['val'].dropna().astype(int).values]

In [ ]:
train = train.reset_index(drop=True)

def generate_prompt(data_point):
    return f"""
    You are an AI assistant specialized in sentiment analysis.
    Classify the sentiment of the following Arabic sentence between square brackets based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral for this Arabic sentence.

    Sentence: [{data_point["Text"]}] = Sentiment: [{data_point["sentiment"]}]
            """.strip()

def generate_test_prompt(data_point):
    return f"""
    You are an AI assistant specialized in sentiment analysis.
    Classify the sentiment of the following Arabic sentence sentence between square brackets based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral for this Arabic sentence.

    Sentence: [{data_point["Text"]}] = Sentiment:
            """.strip()


train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["Text"])
val = pd.DataFrame(val.apply(generate_prompt, axis=1),
                      columns=["Text"])

In [ ]:
y_true = test["sentiment"]
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["Text"])

train_data = Dataset.from_pandas(train)
eval_data = Dataset.from_pandas(train)

# Load Model

In [ ]:
model_name = "QCRI/Fanar-1-9B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/18.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=20,
                temperature=0.2
               )

Device set to use cuda:0


# Predict Without Fine-tuning

In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

# Fine-tune Model

In [ ]:
from trl import SFTConfig

peft_config = LoraConfig(
    lora_alpha=128,
    lora_dropout=0.05,
    r=64,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

training_arguments = SFTConfig(
    output_dir="logs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1, # 4
    optim="paged_adamw_32bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0,
    fp16=False,
    bf16=False,
    max_grad_norm=1,
    max_steps=-1,
    warmup_ratio=0.01,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
    eval_strategy="epoch",
    dataset_text_field="Text",
    max_seq_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

Adding EOS to train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16769 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained("/content/drive/MyDrive/Fanar-ASAD")

It is strongly recommended to train Gemma2 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Epoch,Training Loss,Validation Loss
1,2.372000,2.143942
2,1.867500,1.926715
3,1.823100,1.827770


# Predict

In [ ]:
from peft import PeftModel, PeftConfig
# Load the Lora model
model = PeftModel.from_pretrained(model, '/content/drive/MyDrive/Fanar-ASAD')

In [ ]:
pipe = pipeline(task="text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=1,
                temperature=0.2
               )

Device set to use cuda:0


In [ ]:
y_pred = predict(X_test, model, tokenizer)

In [ ]:
def evaluate(y_true, y_pred):
    labels = ['Positive', 'Negative', 'Neutral']
    mapping = {'Positive': 0, 'Negative': 1, 'Neutral':2, 'none':3}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    f1t = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
    print('\nf1_score: ', f1t)

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred, digits = 4)
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1, 2, 3])
    print('\nConfusion Matrix:')
    print(conf_matrix)

In [4]:
print('Macro Average Scores')
pre = precision_score(y_true=y_true, y_pred=y_pred, average = 'macro')
print(f'Precision = {pre:.3f}')
recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'macro')
print(f'Recall = {recall:.3f}')
f1 = f1_score(y_true=y_true, y_pred=y_pred, average = 'macro')
print(f'F1 score = {f1:.3f}')

print('Weighted Average Scores')
pre = precision_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print(f'Precision = {pre:.3f}')
recall = recall_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print(f'Recall = {recall:.3f}')
f1 = f1_score(y_true=y_true, y_pred=y_pred, average = 'weighted')
print(f'F1 score = {f1:.3f}')

Macro Average Scores
Precision = 0.407
Recall = 0.485
F1 score = 0.371
Weighted Average Scores
Precision = 0.737
Recall = 0.757
F1 score = 0.759


# Pred on Collected data

In [ ]:
collected = pd.read_excel('All Climate Change Data - All Related.xlsx')

collected.drop_duplicates(subset='text', inplace = True)
collected.dropna(inplace = True, subset='text')
collected.reset_index(drop=True, inplace = True)

data.shape

(23957, 2)

In [ ]:
collected['text'] = collected['text'].str.replace(r'[^\w\s]+', '')
collected['text'] = collected['text'].str.replace("\s+", " ", regex=True)

collected['text'] = collected['text'].apply(normalize_arabic)

collected['text'] = data_cleaning(collected['text'])

collected['text'] = collected['text'].apply(remove_ids)

collected.dropna(inplace = True)
collected = collected.drop_duplicates(subset = 'text')
collected = collected.rename(columns={'text': 'Text'})

collected_test = pd.DataFrame(collected.apply(generate_test_prompt, axis=1), columns=["Text"])

In [ ]:
pred = []

max_length = tokenizer.model_max_length
for i in tqdm(range(len(collected_test))):
    prompt = collected_test.iloc[i]["Text"]
    # prompt = prompt[:max_length]
    # result = pipe(prompt, pad_token_id=pipe.tokenizer.eos_token_id)
    result = pipe(prompt, truncation=True, pad_token_id=pipe.tokenizer.eos_token_id)
    answer = result[0]['generated_text'].split("=")[-1].lower()

    if "positive" in answer:
        pred.append("Positive")
    elif "negative" in answer:
        pred.append("Negative")
    elif "neutral" in answer:
        pred.append("Neutral")
    else:
        pred.append('none')

100%|██████████| 8992/8992 [30:44<00:00,  4.88it/s]


In [ ]:
evaluate(collected['Final Label'], pred)

Accuracy: 0.633

f1_score:  0.6314175263921998

Classification Report:
              precision    recall  f1-score   support

           0     0.6365    0.6059    0.6208      2205
           1     0.6732    0.7433    0.7065      3035
           2     0.5942    0.5605    0.5769      3752
           3     0.0000    0.0000    0.0000         0

    accuracy                         0.6333      8992
   macro avg     0.4760    0.4774    0.4761      8992
weighted avg     0.6313    0.6333    0.6314      8992


Confusion Matrix:
[[1336   97  772    0]
 [ 113 2256  664    2]
 [ 650  998 2103    1]
 [   0    0    0    0]]
